In [1]:
"""
STEP 1: KHÁM PHÁ DỮ LIỆU THÔ
==============================
Chạy file này để kiểm tra format thực tế của UAH-DriveSet.
Output: in ra cấu trúc, số dòng, và sample của từng file.
"""

import os
import pandas as pd
from pathlib import Path

# ── Chỉnh đường dẫn đến thư mục UAH-DriveSet ──────────────────────────────
DATA_ROOT = Path("../data/archive/UAH-DRIVESET-v1")
# ──────────────────────────────────────────────────────────────────────────

def explore_file(filepath, n=3):
    """Đọc và in vài dòng đầu của file txt."""
    try:
        with open(filepath, "r") as f:
            lines = f.readlines()
        print(f"\n   {filepath.name}  ({len(lines)} dòng)")
        for i, line in enumerate(lines[:n]):
            print(f"     [{i}] {line.strip()}")
    except Exception as e:
        print(f"   Lỗi đọc {filepath.name}: {e}")


def explore_trip(trip_folder):
    """Khám phá 1 trip folder."""
    print(f"\n{'='*60}")
    print(f" {trip_folder.name}")
    print(f"{'='*60}")

    target_files = [
        "RAW_GPS.txt",
        "RAW_ACCELEROMETERS.txt",
        "EVENTS_INERTIAL.txt",
        "PROC_OPENSTREETMAP_DATA.txt",
        "SEMANTIC_FINAL.txt",
    ]

    for fname in target_files:
        fpath = trip_folder / fname
        if fpath.exists():
            explore_file(fpath, n=3)
        else:
            print(f"\n    {fname} — không tồn tại")


def scan_dataset(root):
    """Quét toàn bộ dataset và in thống kê."""
    print(f"\n{'='*60}")
    print(f" TỔNG QUAN DATASET: {root}")
    print(f"{'='*60}")

    trips = []
    for folder in sorted(root.rglob("*SECONDARY*")) + sorted(root.rglob("*MOTORWAY*")):
        if folder.is_dir() and (folder / "RAW_GPS.txt").exists():
            parts = folder.name.split("-")
            if len(parts) >= 5:
                trips.append({
                    "folder":   folder.name,
                    "driver":   parts[2],
                    "behavior": parts[3],
                    "road":     parts[4],
                    "distance": parts[1],
                })

    df = pd.DataFrame(trips)
    if df.empty:
        print(" Không tìm thấy trip nào. Kiểm tra lại DATA_ROOT.")
        return []

    print(f"\nTổng số trips: {len(df)}")
    print(f"\nPhân bố behavior:")
    print(df["behavior"].value_counts().to_string())
    print(f"\nPhân bố road:")
    print(df["road"].value_counts().to_string())
    print(f"\nPhân bố driver:")
    print(df["driver"].value_counts().to_string())

    return trips


if __name__ == "__main__":
    if not DATA_ROOT.exists():
        print(f" Không tìm thấy: {DATA_ROOT}")
        print("   Hãy chỉnh DATA_ROOT cho đúng đường dẫn.")
    else:
        # 1. Thống kê tổng quan
        trips = scan_dataset(DATA_ROOT)

        # 2. Khám phá 1 trip mẫu (trip đầu tiên tìm được)
        for folder in sorted(DATA_ROOT.rglob("*/")):
            if folder.is_dir() and (folder / "RAW_GPS.txt").exists():
                explore_trip(folder)
                break   # chỉ xem 1 trip để hiểu format



 TỔNG QUAN DATASET: ../data/archive/UAH-DRIVESET-v1

Tổng số trips: 40

Phân bố behavior:
behavior
DROWSY        12
AGGRESSIVE    11
NORMAL         7
NORMAL1        5
NORMAL2        5

Phân bố road:
road
SECONDARY    22
MOTORWAY     18

Phân bố driver:
driver
D1    7
D2    7
D3    7
D4    7
D5    7
D6    5

 20151110175712-16km-D1-NORMAL1-SECONDARY

   RAW_GPS.txt  (624 dòng)
     [0] 7.85 65.2 40.512787 -3.404477 612.7 4 5 331.9 0 0 0 0
     [1] 8.83 64.5 40.512924 -3.404577 612.5 4 5 331.9 0 0 0 0
     [2] 9.82 63.6 40.513065 -3.404680 612.9 4 5 330.8 1.055 0 0 0

   RAW_ACCELEROMETERS.txt  (6170 dòng)
     [0] 6.94 1 0.017 -0.011 0.018 -0.005 0.008 0.018 -1.523 0.015 0.012
     [1] 7.03 1 0.046 0.007 0.019 0.016 -0.002 0.018 -1.522 0.012 0.012
     [2] 7.14 1 0.052 -0.016 0.027 0.037 -0.005 0.018 -1.520 0.014 0.011

   EVENTS_INERTIAL.txt  (3 dòng)
     [0] 26.14 2 1 40.515564 -3.406911 20151110175738
     [1] 206.78 2 1 40.533783 -3.451114 20151110180038
     [2] 458.92 2 1 40.557

In [2]:
"""
STEP 2: FEATURE ENGINEERING
=============================
Đọc từng trip → tính feature vector → gộp thành DataFrame.
Output: raw_features.csv (chưa scale, chưa clean)
"""

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Config ─────────────────────────────────────────────────────────────────
DATA_ROOT  = Path("../data/archive/UAH-DRIVESET-v1")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Ngưỡng phát hiện event (đơn vị: g, 1g = 9.81 m/s²)
HARD_BRAKE_G   = -0.4
HARD_ACCEL_G   = +0.4
SHARP_TURN_G   =  0.4
JERK_THRESH_G  =  0.3
# ──────────────────────────────────────────────────────────────────────────


# ── Parsers ────────────────────────────────────────────────────────────────

def parse_folder_name(folder_name):
    """
    Parse tên folder: '20151111125233-24km-D1-AGGRESSIVE-MOTORWAY'
    Trả về dict với driver, behavior, road, distance_km.
    """
    parts = folder_name.split("-")
    try:
        return {
            "trip_id":       folder_name,
            "driver":        parts[2],
            "behavior":      parts[3].upper(),   # NORMAL / AGGRESSIVE / DROWSY
            "road_type":     0 if "MOTORWAY" in parts[4].upper() else 1,
            "road_name":     parts[4].upper(),
            "distance_label_km": int(re.sub(r"[^0-9]", "", parts[1])),
        }
    except Exception:
        return None


def read_raw_gps(filepath):
    """
    RAW_GPS.txt — format UAH:
    timestamp  lat  lon  altitude  speed(km/h)  bearing  accuracy
    Tần suất: 1Hz
    """
    try:
        df = pd.read_csv(
            filepath, sep=r"\s+", header=None, comment="#",
            names=["timestamp","lat","lon","altitude","speed","bearing","accuracy"]
        )
        df = df.apply(pd.to_numeric, errors="coerce").dropna()
        return df
    except Exception:
        return None


def read_raw_accel(filepath):
    """
    RAW_ACCELEROMETERS.txt — format UAH:
    timestamp  accel_x  accel_y  accel_z  gyro_x  gyro_y  gyro_z
    Đơn vị: m/s² và rad/s, tần suất ~100Hz
    """
    try:
        df = pd.read_csv(
            filepath, sep=r"\s+", header=None, comment="#",
            names=["timestamp","accel_x","accel_y","accel_z",
                   "gyro_x","gyro_y","gyro_z"]
        )
        df = df.apply(pd.to_numeric, errors="coerce").dropna()
        return df
    except Exception:
        return None


def read_events_inertial(filepath):
    """
    EVENTS_INERTIAL.txt — danh sách event đã được UAH detect sẵn:
    timestamp  event_type  intensity
    event_type: 1=hard_accel, 2=hard_brake, 3=turn_right,
                4=turn_left, 5=weaving, 6=overspeeding
    """
    try:
        df = pd.read_csv(
            filepath, sep=r"\s+", header=None, comment="#",
            names=["timestamp","event_type","intensity"]
        )
        df = df.apply(pd.to_numeric, errors="coerce").dropna()
        return df
    except Exception:
        return pd.DataFrame(columns=["timestamp","event_type","intensity"])


def read_osm_data(filepath):
    """
    PROC_OPENSTREETMAP_DATA.txt:
    timestamp  speed_limit  road_type  ...
    """
    try:
        df = pd.read_csv(
            filepath, sep=r"\s+", header=None, comment="#",
        )
        df = df.apply(pd.to_numeric, errors="coerce").dropna()
        return df
    except Exception:
        return None


# ── Feature computation ────────────────────────────────────────────────────

def compute_gps_features(gps):
    """Tính features từ RAW_GPS."""
    if gps is None or len(gps) < 5:
        return {}

    speed = gps["speed"].clip(lower=0)   # bỏ giá trị âm (GPS noise)
    duration_s  = len(speed)             # 1Hz → số giây
    distance_km = (speed.mean() * duration_s) / 3600

    speed_diff = speed.diff().abs().dropna()

    return {
        # Tốc độ
        "speed_mean":       speed.mean(),
        "speed_std":        speed.std(),
        "speed_max":        speed.max(),
        "speed_p95":        speed.quantile(0.95),
        "speed_cv":         speed.std() / (speed.mean() + 1e-6),   # coeff of variation

        # Biến động tốc độ
        "speed_change_mean": speed_diff.mean(),
        "speed_change_max":  speed_diff.max(),

        # Trip info
        "duration_min":     duration_s / 60,
        "distance_km":      distance_km,
    }


def compute_accel_features(imu, distance_km):
    """Tính features từ RAW_ACCELEROMETERS."""
    if imu is None or len(imu) < 10:
        return {}

    # Chuyển m/s² → g
    ax = imu["accel_x"] / 9.81   # gia tốc dọc
    ay = imu["accel_y"] / 9.81   # gia tốc ngang
    az = imu["accel_z"] / 9.81   # gia tốc thẳng đứng

    gz = imu["gyro_z"]            # tốc độ xoay (rad/s)

    # Jerk = Δaccel / Δtime  (xấp xỉ với diff)
    dt = imu["timestamp"].diff().median() / 1000  # ms → s
    dt = dt if dt > 0 else 0.01
    jerk_x = ax.diff().abs() / dt
    jerk_y = ay.diff().abs() / dt

    # Magnitude
    accel_mag = np.sqrt(ax**2 + ay**2 + az**2)

    # Event counts
    n_hard_brake  = (ax < HARD_BRAKE_G).sum()
    n_hard_accel  = (ax > HARD_ACCEL_G).sum()
    n_sharp_turn  = (ay.abs() > SHARP_TURN_G).sum()
    n_jerk_event  = (jerk_x > JERK_THRESH_G).sum()

    dist = max(distance_km, 0.1)

    return {
        # Gia tốc dọc
        "accel_long_mean":     ax.mean(),
        "accel_long_std":      ax.std(),
        "accel_long_max":      ax.max(),
        "accel_long_min":      ax.min(),   # âm mạnh = phanh gấp

        # Gia tốc ngang
        "accel_lat_mean":      ay.abs().mean(),
        "accel_lat_std":       ay.std(),
        "accel_lat_max":       ay.abs().max(),

        # Gyroscope
        "gyro_z_mean":         gz.abs().mean(),
        "gyro_z_max":          gz.abs().max(),
        "gyro_z_std":          gz.std(),

        # Magnitude
        "accel_mag_mean":      accel_mag.mean(),
        "accel_mag_std":       accel_mag.std(),
        "rms_accel":           np.sqrt((accel_mag**2).mean()),

        # Jerk
        "jerk_long_mean":      jerk_x.mean(),
        "jerk_long_max":       jerk_x.max(),
        "jerk_lat_max":        jerk_y.max(),
        "rms_jerk":            np.sqrt((jerk_x**2 + jerk_y**2).mean()),

        # Event counts (raw)
        "hard_brake_count":    int(n_hard_brake),
        "hard_accel_count":    int(n_hard_accel),
        "sharp_turn_count":    int(n_sharp_turn),
        "jerk_event_count":    int(n_jerk_event),

        # Event rate per km (normalized) ← QUAN TRỌNG
        "hard_brake_per_km":   n_hard_brake  / dist,
        "hard_accel_per_km":   n_hard_accel  / dist,
        "sharp_turn_per_km":   n_sharp_turn  / dist,
        "aggressive_event_rate": (n_hard_brake + n_hard_accel + n_sharp_turn) / dist,
    }


def compute_event_features(events, distance_km):
    """Tính features từ EVENTS_INERTIAL (UAH pre-detected events)."""
    if events is None or events.empty:
        return {
            "uah_hard_accel":    0,
            "uah_hard_brake":    0,
            "uah_turn_right":    0,
            "uah_turn_left":     0,
            "uah_weaving":       0,
            "uah_overspeeding":  0,
            "uah_total_events":  0,
            "uah_event_rate":    0,
            "uah_intensity_mean": 0,
            "uah_intensity_max":  0,
        }

    dist = max(distance_km, 0.1)
    # event_type: 1=accel, 2=brake, 3=right, 4=left, 5=weaving, 6=overspeed
    etype = events["event_type"]
    total = len(events)

    return {
        "uah_hard_accel":    (etype == 1).sum(),
        "uah_hard_brake":    (etype == 2).sum(),
        "uah_turn_right":    (etype == 3).sum(),
        "uah_turn_left":     (etype == 4).sum(),
        "uah_weaving":       (etype == 5).sum(),
        "uah_overspeeding":  (etype == 6).sum(),
        "uah_total_events":  total,
        "uah_event_rate":    total / dist,
        "uah_intensity_mean": events["intensity"].mean() if total > 0 else 0,
        "uah_intensity_max":  events["intensity"].max()  if total > 0 else 0,
    }


# Main

def process_trip(trip_folder):
    """Xử lý 1 trip → trả về 1 dict features."""

    # Parse metadata từ tên folder
    meta = parse_folder_name(trip_folder.name)
    if meta is None:
        return None

    # Load files
    gps    = read_raw_gps(trip_folder / "RAW_GPS.txt")
    imu    = read_raw_accel(trip_folder / "RAW_ACCELEROMETERS.txt")
    events = read_events_inertial(trip_folder / "EVENTS_INERTIAL.txt")

    # Tính GPS features trước để có distance_km
    gps_feats   = compute_gps_features(gps)
    distance_km = gps_feats.get("distance_km", meta["distance_label_km"])

    # Tính các nhóm feature
    accel_feats = compute_accel_features(imu, distance_km)
    event_feats = compute_event_features(events, distance_km)

    # Context features
    context_feats = {
        "road_type":  meta["road_type"],   # 0=motorway, 1=secondary
    }

    # Gộp tất cả
    row = {}
    row.update(meta)
    row.update(gps_feats)
    row.update(accel_feats)
    row.update(event_feats)
    row.update(context_feats)

    return row


def find_all_trips(root):
    """Tìm tất cả trip folders có RAW_GPS.txt."""
    trips = []
    for folder in sorted(root.rglob("*/")):
        if folder.is_dir() and (folder / "RAW_GPS.txt").exists():
            # Kiểm tra tên folder đúng format
            parts = folder.name.split("-")
            if len(parts) >= 5:
                trips.append(folder)
    return trips


if __name__ == "__main__":
    print(" STEP 2: Feature Engineering")
    print(f"   Data root: {DATA_ROOT}")

    if not DATA_ROOT.exists():
        print(f"Không tìm thấy: {DATA_ROOT}")
        exit(1)

    trip_folders = find_all_trips(DATA_ROOT)
    print(f"   Tìm thấy {len(trip_folders)} trips\n")

    if len(trip_folders) == 0:
        print(" Không có trip nào. Kiểm tra DATA_ROOT.")
        exit(1)

    # Xử lý từng trip
    all_rows = []
    errors   = []

    for trip_folder in tqdm(trip_folders, desc="Processing trips"):
        try:
            row = process_trip(trip_folder)
            if row:
                all_rows.append(row)
        except Exception as e:
            errors.append((trip_folder.name, str(e)))

    if errors:
        print(f"\n  {len(errors)} trips bị lỗi:")
        for name, err in errors:
            print(f"   {name}: {err}")

    # Tạo DataFrame
    df = pd.DataFrame(all_rows)
    print(f"\n Đã xử lý {len(df)} trips")
    print(f"   Shape: {df.shape}")
    print(f"\nPhân bố behavior:")
    print(df["behavior"].value_counts().to_string())

    # Lưu
    out_path = OUTPUT_DIR / "raw_features.csv"
    df.to_csv(out_path, index=False)
    print(f"\n Đã lưu: {out_path}")

 STEP 2: Feature Engineering
   Data root: ../data/archive/UAH-DRIVESET-v1
   Tìm thấy 40 trips



Processing trips: 100%|██████████| 40/40 [00:00<00:00, 58.86it/s]


 Đã xử lý 40 trips
   Shape: (40, 50)

Phân bố behavior:
behavior
DROWSY        12
AGGRESSIVE    11
NORMAL         7
NORMAL1        5
NORMAL2        5

 Đã lưu: output/raw_features.csv


In [ ]:
import pandas as pd
from pathlib import Path

# --- CẤU HÌNH TỐI GIẢN ---
INPUT_FILE = Path("output/raw_features.csv")
OUTPUT_FILE = Path("output/dataset_final_v2.csv")

# Chỉ lấy những cột "gốc" khó sai nhất
CORE_FEATURES = [
    'speed_mean', 'speed_std', 
    'accel_long_std', 'accel_lat_std', 
    'distance_km'
]
META = ['trip_id', 'behavior', 'label']

if INPUT_FILE.exists():
    df_raw = pd.read_csv(INPUT_FILE)
    
    # Gán nhãn lại
    mapping = {"NORMAL": 0, "NORMAL1": 0, "NORMAL2": 0, "AGGRESSIVE": 1, "DROWSY": 2}
    df_raw['label'] = df_raw['behavior'].str.strip().map(mapping)
    
    df_final = df_raw[META + CORE_FEATURES].copy()
    
    # Xử lý NaN (nếu có) bằng Median cho an toàn
    df_final = df_final.fillna(df_final.median(numeric_only=True))
    
    # Lưu 
    df_final.to_csv(OUTPUT_FILE, index=False)
    
    print(f" Đã tái cấu trúc thành công {len(df_final)} trips.")
    print(f" Các đặc trưng hiện có: {CORE_FEATURES}")
    print(df_final.head(3))
else:
    print(" Không tìm thấy file nguồn!")

 Đã tái cấu trúc thành công 40 trips.
 Các đặc trưng hiện có: ['speed_mean', 'speed_std', 'accel_long_std', 'accel_lat_std', 'distance_km']
                                    trip_id behavior  label  speed_mean  \
0  20151110175712-16km-D1-NORMAL1-SECONDARY  NORMAL1      0    0.363782   
1  20151110180824-16km-D1-NORMAL2-SECONDARY  NORMAL2      0    0.452951   
2    20151111123124-25km-D1-NORMAL-MOTORWAY   NORMAL      0    0.165509   

   speed_std  accel_long_std  accel_lat_std  distance_km  
0   0.631467        0.002490       0.003191     0.063056  
1   0.677528        0.002400       0.003172     0.078889  
2   0.483005        0.002512       0.004966     0.039722  


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Vẽ biểu đồ phân tán giữa Gia tốc dọc và Gia tốc ngang
# Đây là cách tốt nhất để thấy sự khác biệt giữa các nhóm hành vi
scatter = sns.scatterplot(
    data=df_final, 
    x='accel_long_std', 
    y='accel_lat_std', 
    hue='behavior', 
    style='behavior',
    s=100,
    palette='viridis'
)

plt.title('BẢN ĐỒ HÀNH VI LÁI XE', fontsize=14)
plt.xlabel('Độ gắt Phanh/Ga (Accel Long Std)', fontsize=12)
plt.ylabel('Độ gắt Vào cua (Accel Lat Std)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()